In [4]:
"""
hls4ml_convert.py
=================
Convert the trained Keras CNN to an HLS design using hls4ml,
then compare resource usage and latency against the hand-written VHDL.

Prerequisites:
    pip install hls4ml[profiling] tensorflow numpy

Steps performed:
    1. Rebuild and retrain the Keras model (same architecture as cnn_top.vhd)
    2. Configure hls4ml with fixed-point precision
    3. Generate HLS project targeting xc7z020clg484-1
    4. (Optional) Run Vivado HLS synthesis via hls4ml
    5. Print comparison table: VHDL vs hls4ml

Run:
    python hls4ml_convert.py

Output folder:
    hls4ml_prj/   (HLS project files, reports)
"""

import os
import numpy as np
import tensorflow as tf
from tensorflow import keras

# -- hls4ml ------------------------------------------------------
try:
    import hls4ml
except ImportError:
    raise ImportError(
        "hls4ml not found. Install with:\n"
        "    pip install hls4ml[profiling]"
    )

# ----------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------
EPOCHS        = 10
BATCH_SIZE    = 128
HLS_PRJ_DIR   = os.path.join(os.getcwd(), "hls4ml_prj")
PART          = "xc7z020clg484-1"
CLOCK_PERIOD  = 10          # ns  (100 MHz, same as VHDL design)
REUSE_FACTOR  = 1           # 1 = maximum parallelism (closest to VHDL)

# Fixed-point precision
#
# Overflow analysis for this CNN with normalised [0,1] inputs:
#   conv1 : 3x3x1  =  9 MACs,  max output ~  9 x 1 x 1  =    9
#   conv2 : 3x3x8  = 72 MACs,  max output ~ 72 x 9 x 1  =  648
#   fc1   : 400 inputs,        max output ~ 400 x 5 x 1  = 2000 (typical << max)
#
# ap_fixed<16,6>  range [-32,  32) -> clips conv2 -> 10% accuracy (random!)
# ap_fixed<16,10> range [-512, 512) -> safely covers all layers
#
W_PRECISION   = "ap_fixed<8,1>"     # 8-bit weights  (matches VHDL WEIGHT_WIDTH=8)
A_PRECISION   = "ap_fixed<16,10>"   # 16-bit, 10 integer bits -> range [-512,512)


# ----------------------------------------------------------------
# Build and train model (identical architecture to cnn_top.vhd)
# ----------------------------------------------------------------
def build_and_train():
    print("=" * 60)
    print("Loading MNIST and training model")
    print("=" * 60)

    (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
    x_train = x_train.astype(np.float32) / 255.0
    x_test  = x_test.astype(np.float32)  / 255.0
    x_train = x_train[..., np.newaxis]
    x_test  = x_test[..., np.newaxis]

    model = keras.Sequential([
        keras.layers.Input(shape=(28, 28, 1)),
        # Conv1: 8 filters, 3x3, valid -> 26x26x8
        keras.layers.Conv2D(8,  (3,3), padding='valid', activation='relu', name='conv1'),
        keras.layers.MaxPooling2D((2,2), name='pool1'),          # -> 13x13x8
        # Conv2: 16 filters, 3x3, valid -> 11x11x16
        keras.layers.Conv2D(16, (3,3), padding='valid', activation='relu', name='conv2'),
        keras.layers.MaxPooling2D((2,2), name='pool2'),          # -> 5x5x16
        keras.layers.Flatten(name='flatten'),                    # -> 400
        keras.layers.Dense(64, activation='relu', name='fc1'),   # -> 64
        keras.layers.Dense(10, activation='softmax', name='fc2'),# -> 10
    ], name='cnn_mnist')

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    model.summary()

    model.fit(x_train, y_train,
              epochs=EPOCHS,
              batch_size=BATCH_SIZE,
              validation_split=0.1,
              verbose=1)

    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"\nTest accuracy: {acc*100:.2f}%  Loss: {loss:.4f}")

    return model, x_test, y_test


# ----------------------------------------------------------------
# Configure hls4ml
# ----------------------------------------------------------------
def configure_hls4ml(model):
    print()
    print("=" * 60)
    print("Configuring hls4ml")
    print("=" * 60)

    config = hls4ml.utils.config_from_keras_model(
        model,
        granularity='name',
        default_precision=W_PRECISION,
        default_reuse_factor=REUSE_FACTOR
    )

    # -- Per-layer precision -------------------------------------
    for layer in ['conv1', 'conv2', 'fc1', 'fc2']:
        config['LayerName'][layer]['Precision']['weight']     = W_PRECISION
        config['LayerName'][layer]['Precision']['bias']       = 'ap_fixed<16,6>'
        config['LayerName'][layer]['Precision']['result']     = A_PRECISION
        config['LayerName'][layer]['ReuseFactor']             = REUSE_FACTOR

    for layer in ['pool1', 'pool2']:
        config['LayerName'][layer]['Precision']['result']     = A_PRECISION

    # -- Auto-generated activation layers ------------------------
    # hls4ml splits Conv2D+ReLU into e.g. conv1 + conv1_relu.
    # Use unsigned for relu (output always >= 0).
    # ap_ufixed<16,10>: unsigned, 10 integer bits -> range [0, 1024).
    RELU_PREC = "ap_ufixed<16,10>"
    for layer in ['conv1_relu', 'conv2_relu', 'fc1_relu']:
        config['LayerName'][layer]['Precision']['result']     = RELU_PREC

    # Softmax output is in [0,1]
    config['LayerName']['fc2_softmax']['Precision']['result'] = 'ap_fixed<16,6>'

    # Flatten carries pool2 outputs unchanged
    if 'flatten' in config['LayerName']:
        config['LayerName']['flatten']['Precision']['result'] = A_PRECISION

    # Input layer name increments each Keras session (input_1, input_2 ...).
    # Pixels in [0,1]: use unsigned 16-bit matching A_PRECISION integer range.
    for layer_name in list(config['LayerName'].keys()):
        if layer_name.startswith('input_'):
            config['LayerName'][layer_name]['Precision']['result'] = 'ap_ufixed<16,10>'

    print("Layer configuration:")
    for name, cfg in config['LayerName'].items():
        prec = cfg.get('Precision', {})
        rf   = cfg.get('ReuseFactor', '-')
        print(f"  {name:12s}  weight={prec.get('weight','default'):15s}  "
              f"result={prec.get('result','default'):15s}  RF={rf}")

    return config


# ----------------------------------------------------------------
# Convert model to HLS
# ----------------------------------------------------------------
def convert_to_hls(model, config):
    print()
    print("=" * 60)
    print("Converting model to HLS")
    print("=" * 60)

    # Remove stale project to avoid using a cached compiled binary from a
    # previous run with different precision (causes ~10% accuracy).
    import shutil
    if os.path.exists(HLS_PRJ_DIR):
        shutil.rmtree(HLS_PRJ_DIR)
        print(f"Removed stale HLS project: {HLS_PRJ_DIR}")

    hls_model = hls4ml.converters.convert_from_keras_model(
        model,
        hls_config=config,
        output_dir=HLS_PRJ_DIR,
        backend='Vivado',
        part=PART,
        clock_period=CLOCK_PERIOD,
        io_type='io_parallel'
    )

    print(f"HLS project written to: {HLS_PRJ_DIR}")
    hls_model.compile()           # compile C-sim model fresh

    return hls_model


# ----------------------------------------------------------------
# Validate HLS model accuracy vs Keras model
# ----------------------------------------------------------------
def validate(model, hls_model, x_test, y_test, n=1000):
    print()
    print("=" * 60)
    print(f"Validating HLS model accuracy (first {n} test samples)")
    print("=" * 60)

    x_sample = x_test[:n]
    y_sample = y_test[:n]

    keras_preds = np.argmax(model.predict(x_sample, verbose=0), axis=1)
    keras_acc   = np.mean(keras_preds == y_sample)

    # NOTE: any "bound cannot be negative" warnings from hls4ml C-sim
    # are informational only and do not affect correctness.
    hls_preds   = np.argmax(hls_model.predict(x_sample), axis=1)
    hls_acc     = np.mean(hls_preds == y_sample)

    match_rate  = np.mean(keras_preds == hls_preds)

    print(f"  Keras  accuracy : {keras_acc*100:.2f}%")
    print(f"  hls4ml accuracy : {hls_acc*100:.2f}%")
    print(f"  Prediction match: {match_rate*100:.2f}%")

    if match_rate < 0.95:
        print("  WARNING: match rate below 95% - check precision config")
    else:
        print("  OK: high match rate - precision config is good")

    return keras_acc, hls_acc


# ----------------------------------------------------------------
# Helper: find Vivado/Vitis HLS and add its bin dir to PATH
# ----------------------------------------------------------------
def _find_vivado_and_patch_path():
    import shutil, glob

    for tool in ('vivado_hls', 'vitis_hls'):
        if shutil.which(tool):
            return tool

    roots = ['/opt/Xilinx', '/tools/Xilinx',
             os.path.expanduser('~/Xilinx'), 'C:/Xilinx']
    patterns = [
        '*/Vitis_HLS/*/bin',
        '*/Vivado_HLS/*/bin',
        '*/Vivado/*/bin',
        '*/Vitis/*/bin',
        '*/Vitis_HLS/bin',
        '*/Vivado/bin',
    ]
    candidates = []
    for root in roots:
        for pat in patterns:
            candidates += glob.glob(os.path.join(root, pat))

    def _version_key(p):
        for good in ('2023.2', '2022.2', '2022.1', '2024.1',
                     '2021.2', '2021.1', '2020.2', '2020.1'):
            if good in p:
                return good
        return p

    candidates.sort(key=_version_key, reverse=True)

    for bindir in candidates:
        for tool in ('vitis_hls', 'vivado_hls'):
            exe = os.path.join(bindir, tool)
            if os.path.isfile(exe) or os.path.isfile(exe + '.bat'):
                os.environ['PATH'] = bindir + os.pathsep + os.environ.get('PATH', '')
                print(f"  Added to PATH: {bindir}")
                print(f"  Tool found   : {tool}")
                return tool
    return None


# ----------------------------------------------------------------
# Run Vivado HLS synthesis (requires Vivado/Vitis HLS installed)
# ----------------------------------------------------------------
def run_synthesis(hls_model):
    print()
    print("=" * 60)
    print("Running Vivado HLS synthesis (this may take 10-20 minutes)")
    print("=" * 60)

    import shutil
    tool = _find_vivado_and_patch_path()

    if tool is None:
        print()
        print("  ERROR: vivado_hls / vitis_hls not found.")
        print()
        print("  Fix: add Vivado bin to PATH in a notebook cell before calling:")
        print("    import os")
        print("    os.environ['PATH'] = '/opt/Xilinx/Vitis_HLS/2022.2/bin:' \\")
        print("                         + os.environ['PATH']")
        return False

    # Vivado 2020.2+ ships 'vitis_hls' only; hls4ml hard-codes 'vivado_hls'.
    # Create a thin shim so it resolves.
    if tool == 'vitis_hls' and not shutil.which('vivado_hls'):
        vitis_path = shutil.which('vitis_hls')
        shim_dir   = os.path.join(os.path.dirname(vitis_path), '_hls4ml_shim')
        os.makedirs(shim_dir, exist_ok=True)
        shim = os.path.join(shim_dir, 'vivado_hls')
        if not os.path.exists(shim):
            with open(shim, 'w') as f:
                f.write(f'#!/bin/sh\nexec "{vitis_path}" "$@"\n')
            os.chmod(shim, 0o755)
        os.environ['PATH'] = shim_dir + os.pathsep + os.environ['PATH']
        print("  Created vivado_hls -> vitis_hls shim for hls4ml compatibility.")

    try:
        hls_model.build(csim=False, synth=True, vsynth=False)
        print("Synthesis complete.")
        return True
    except Exception as e:
        print(f"  Synthesis failed: {e}")
        return False


# ----------------------------------------------------------------
# Estimate resources without Vivado (counts MACs -> DSP estimate)
# ----------------------------------------------------------------
def estimate_resources(model):
    print()
    print("=" * 60)
    print("Resource Estimate (no Vivado required)")
    print("=" * 60)
    print()

    total_macs = 0
    rows = []

    for layer in model.layers:
        cfg   = layer.get_config()
        name  = layer.name
        ltype = layer.__class__.__name__

        if ltype == 'Conv2D':
            out_h   = layer.output_shape[1]
            out_w   = layer.output_shape[2]
            filters = cfg['filters']
            kh, kw  = cfg['kernel_size']
            in_ch   = layer.input_shape[-1]
            macs    = out_h * out_w * filters * kh * kw * in_ch
            total_macs += macs
            rows.append((name, ltype, f"{out_h}x{out_w}x{filters}", macs))

        elif ltype == 'Dense':
            in_f  = layer.input_shape[-1]
            out_f = layer.output_shape[-1]
            macs  = in_f * out_f
            total_macs += macs
            rows.append((name, ltype, f"{in_f}->{out_f}", macs))

    print(f"  {'Layer':<12} {'Type':<12} {'Shape':<14} {'MACs':>10}")
    print("  " + "-" * 52)
    for name, ltype, shape, macs in rows:
        print(f"  {name:<12} {ltype:<12} {shape:<14} {macs:>10,}")
    print("  " + "-" * 52)
    print(f"  {'TOTAL':<12} {'':<12} {'':<14} {total_macs:>10,}")
    print()
    print(f"  DSP48E1 estimate (RF=1, fully unrolled): ~{total_macs:,}")
    print(f"  Custom VHDL actual                     :  150")
    print()
    print("  hls4ml Resource strategy shares DSPs across filters,")
    print("  so actual count will be lower. Run synthesis for exact numbers.")
    return total_macs


# ----------------------------------------------------------------
# Print comparison table
# ----------------------------------------------------------------
def print_comparison(hls_model):
    print()
    print("=" * 60)
    print("Resource Comparison: Custom VHDL vs hls4ml")
    print("=" * 60)

    vhdl = {
        "DSP48E1" : 150,
        "RAMB36E1": 2,
        "RAMB18E1": 1,
        "LUT"     : 42474,
        "FF"      : 6574,
        "Fmax_MHz": 102.6,
    }

    print(f"\n{'Resource':<12} {'Custom VHDL':>14} {'hls4ml':>14} {'Device Max':>14}")
    print("-" * 56)
    print(f"{'DSP48E1':<12} {vhdl['DSP48E1']:>14}  {'(see rpt)':>13}  {'220':>13}")
    print(f"{'RAMB36E1':<12} {vhdl['RAMB36E1']:>14}  {'(see rpt)':>13}  {'140':>13}")
    print(f"{'RAMB18E1':<12} {vhdl['RAMB18E1']:>14}  {'(see rpt)':>13}  {'280':>13}")
    print(f"{'LUT':<12} {vhdl['LUT']:>14}  {'(see rpt)':>13}  {'53200':>13}")
    print(f"{'FF':<12} {vhdl['FF']:>14}  {'(see rpt)':>13}  {'106400':>13}")
    print(f"{'Fmax (MHz)':<12} {vhdl['Fmax_MHz']:>14.1f}  {'(see rpt)':>13}  {'(target 100)':>13}")

    print()
    print("To fill in hls4ml column:")
    print("  1. Run run_synthesis(hls_model)")
    print(f"  2. Check: {HLS_PRJ_DIR}/myproject_prj/solution1/syn/report/")
    print()

    rpt_path = os.path.join(HLS_PRJ_DIR,
        "myproject_prj", "solution1", "syn", "report",
        "myproject_csynth.rpt")
    if os.path.exists(rpt_path):
        print("Synthesis report found:")
        with open(rpt_path) as f:
            print(f.read())
    else:
        print("(Synthesis not yet run)")


# ----------------------------------------------------------------
# Main
# ----------------------------------------------------------------
if __name__ == "__main__":
    print()
    print("hls4ml CNN Conversion and Comparison")
    print("=====================================")
    print(f"Target device  : {PART}")
    print(f"Clock period   : {CLOCK_PERIOD} ns  ({1000//CLOCK_PERIOD} MHz)")
    print(f"Weight prec    : {W_PRECISION}")
    print(f"Activation prec: {A_PRECISION}")
    print(f"Reuse factor   : {REUSE_FACTOR}")
    print(f"HLS output dir : {HLS_PRJ_DIR}")
    print()

    # Step 1: Train model
    model, x_test, y_test = build_and_train()

    # Step 2: Configure hls4ml
    config = configure_hls4ml(model)

    # Step 3: Convert to HLS (cleans stale project automatically)
    hls_model = convert_to_hls(model, config)

    # Step 4: Validate accuracy
    keras_acc, hls_acc = validate(model, hls_model, x_test, y_test)

    # Step 5a: Estimate resources (no Vivado needed)
    estimate_resources(model)

    # Step 5b: Run Vivado HLS synthesis (uncomment when Vivado is on PATH)
    run_synthesis(hls_model)

    # Step 6: Print comparison table
    print_comparison(hls_model)

    print()
    print("=" * 60)
    print("DONE")
    print("=" * 60)
    print("Next: uncomment run_synthesis(hls_model) and re-run to get")
    print("      full resource report for comparison with VHDL design.")



hls4ml CNN Conversion and Comparison
Target device  : xc7z020clg484-1
Clock period   : 10 ns  (100 MHz)
Weight prec    : ap_fixed<8,1>
Activation prec: ap_fixed<16,10>
Reuse factor   : 1
HLS output dir : /home/mas47153/hls4ml_prj

Loading MNIST and training model
Model: "cnn_mnist"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1 (Conv2D)              (None, 26, 26, 8)         80        
                                                                 
 pool1 (MaxPooling2D)        (None, 13, 13, 8)         0         
                                                                 
 conv2 (Conv2D)              (None, 11, 11, 16)        1168      
                                                                 
 pool2 (MaxPooling2D)        (None, 5, 5, 16)          0         
                                                                 
 flatten (Flatten)           (None, 400)               0

OSError: [Errno 30] Read-only file system: '/opt/Xilinx/Vitis_HLS/2020.2/bin/_hls4ml_shim'